# The attribution referee — GRPO v2 twins on the held-out exam (seed 3407)

**Runtime: L4 — MANDATORY.** ~1h total, ~8-10 units, checkpointed per twin.

The dev verdict (nb 17) leaned "signal causal": real 76.2/93.3 vs random
73.8/86.7 vs init 73.5/91.7 on the 60 new-dev bugs. But the p@1 gap sits
inside dev noise (±3-4) and this is the ending we'd *prefer* — so the frozen
exam decides. One 164×20 run per twin, pure peft merging, pinned harness.

Same-seed paired comparison: init, real, and random all share seed 3407
lineage (A3), so the deltas are twin-vs-twin, not seed lottery.

In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch
assert torch.cuda.is_bf16_supported(), 'Switch runtime to L4.'
name = torch.cuda.get_device_name(0)
assert 'L4' in name, f'Protocol pins the exam to L4; this is {name}.'
print(name, '- OK')

In [ ]:
# 2) Drive + runs
from google.colab import drive
drive.mount('/content/drive')
import os, json, gc
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
KINDS = ['real', 'random']
def met_path(kind):
    return f'{PHASE8}/metrics_grpov2_{kind}_s3407.json'
todo = [k for k in KINDS if not os.path.exists(met_path(k))]
print('to run:', todo)

In [ ]:
%%capture
!pip install -q peft
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# 3) Two-stage merge (pure peft).
# The twin adapters were trained on top of the merged SFT v2 saved at
# /content/merged_sftv2_s3407 — their adapter_config points there, so that
# exact path must exist (stage A) before a twin adapter can be applied (stage B).
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE_ID = 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
MERGED = '/content/merged_sftv2_s3407'
if todo and not os.path.exists(MERGED):
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(base, f'{PHASE8}/sft_v2_s3407_ep2')
    m = m.merge_and_unload()
    m.save_pretrained(MERGED, safe_serialization=True)
    try:
        tok = AutoTokenizer.from_pretrained(f'{PHASE8}/sft_v2_s3407_ep2')
    except Exception:
        tok = AutoTokenizer.from_pretrained(BASE_ID)
    tok.save_pretrained(MERGED)
    del base, m; gc.collect(); torch.cuda.empty_cache()
    print('stage A: merged SFT v2 s3407 rebuilt')
for kind in todo:
    out = f'/content/final_grpov2_{kind}'
    if os.path.exists(out):
        continue
    sft = AutoModelForCausalLM.from_pretrained(MERGED, torch_dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(sft, f'{PHASE8}/grpo_v2_{kind}_s3407')
    m = m.merge_and_unload()
    m.save_pretrained(out, safe_serialization=True)
    try:
        tok = AutoTokenizer.from_pretrained(f'{PHASE8}/grpo_v2_{kind}_s3407')
    except Exception:
        tok = AutoTokenizer.from_pretrained(MERGED)
    tok.save_pretrained(out)
    del sft, m; gc.collect(); torch.cuda.empty_cache()
    print('merged twin:', kind)
print('merges done')

In [ ]:
# 4) Harness at the frozen commit
FROZEN_COMMIT = '8fc5bae6479c4fbbb28c3f8b644f6a15b3f3b5bd'
%cd /content
!rm -rf bigcode-evaluation-harness
!git clone -q https://github.com/bigcode-project/bigcode-evaluation-harness.git
%cd /content/bigcode-evaluation-harness
!git checkout -q {FROZEN_COMMIT}
ver = !git rev-parse HEAD
assert ver[0] == FROZEN_COMMIT
!pip install -q -e .
!pip install -q -U transformers accelerate
!pip uninstall -y -q torchao torchaudio torchvision timm
import transformers
print('harness pinned', ver[0][:8], '| transformers', transformers.__version__)

In [ ]:
# 5) THE RUNS — one exam per twin
TASK = 'humanevalfixtests-python'
PROMPT = 'instruct'
BATCH = 16
%cd /content/bigcode-evaluation-harness
for kind in todo:
    print('=' * 70)
    print(f'GRPO v2 [{kind}] s3407: 164 x 20')
    gen_path = f'{PHASE8}/gens_grpov2_{kind}_s3407.json'
    m_path = met_path(kind)
    model_dir = f'/content/final_grpov2_{kind}'
    !accelerate launch main.py \
      --model {model_dir} --tasks {TASK} --prompt {PROMPT} \
      --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 20 \
      --batch_size {BATCH} --precision bf16 \
      --max_length_generation 2048 --allow_code_execution \
      --save_generations --save_generations_path {gen_path} \
      --metric_output_path {m_path}
    print(open(m_path).read())

In [ ]:
# 6) THE ATTRIBUTION TABLE
def read_m(p):
    m = json.load(open(p)); k = [x for x in m if x != 'config'][0]
    return m[k].get('pass@1', 0) * 100, m[k].get('pass@10', 0) * 100
sft_p = f'{PHASE8}/metrics_sftv2_ep2_s3407.json'
sft1, sft10 = read_m(sft_p) if os.path.exists(sft_p) else (34.51, 47.24)
print(f"{'model (all seed 3407)':<26} pass@1    pass@10")
print(f"{'base':<26}  17.59%    23.50%")
print(f"{'OctoCoder-16B (target)':<26}  30.40%       --")
print(f"{'SFT v2 ep2 (the init)':<26} {sft1:6.2f}%   {sft10:6.2f}%")
rows = {}
for kind in KINDS:
    p = met_path(kind)
    if os.path.exists(p):
        p1, p10 = read_m(p); rows[kind] = (p1, p10)
        print(f"{'GRPO v2 ' + kind.upper():<26} {p1:6.2f}%   {p10:6.2f}%")
    else:
        print(f'GRPO v2 {kind}: not run')
print(f"{'GPT-4 (paper reference)':<26}  ~47.00%  (pass@1 -- NEVER compare our pass@10 to this)")
if len(rows) == 2:
    r1, r10 = rows['real']; n1, n10 = rows['random']
    print(f'\nreal - init:   {r1 - sft1:+.2f} p@1   {r10 - sft10:+.2f} p@10')
    print(f'random - init: {n1 - sft1:+.2f} p@1   {n10 - sft10:+.2f} p@10')
    print(f'real - random: {r1 - n1:+.2f} p@1   {r10 - n10:+.2f} p@10')
    print('\nPre-registered readings: real >> random = signal causal on harder data;')
    print('real ~ random > SFT = process effect persists; both ~ SFT = RL saturated.')
    print('Per-problem McNemar from the saved gens is the stats notebook, not here.')
print('\nBring the whole printout back.')

## Bring back to the session
The **attribution table** with all three deltas — it picks LinkedIn post #2's
ending and closes the v2 attribution question. Also still owed from nb 17:
gate line, reward rows, both probe lines, and the watch-file heads/tails
(`pen_applied` trend on the real run).